# Master — KIČMA v2: multisekvencno, SA i BEZ registracije

**isti pacijenti, ista podela (fold 0), isti trener, ista T2 i maska**; razlika je samo u kanalu 1:

| krak | kanal 0 | kanal 1 | gde |
|---|---|---|---|
| **v1** (već istreniran) | T2 | — | `Dataset011_Spine` |
| **NOREG** | T2 | dodatna sekvenca **bez registracije** (samo preuzorkovana na T2 mrežu) | `Dataset013_SpineNOREG` |
| **REG** | T2 | dodatna sekvenca **registrovana** na T2 (autorska registracija) | `Dataset012_SpineREG` |

- v1 → NOREG = koliko donosi **dodatna sekvenca**; NOREG → REG = koliko donosi **registracija** (uticaj na kvalitet detekcije, obećano u radu).
- Trener je **isti kao za v1** (podrazumevani LR), samo čuva checkpoint na 10 epoha → poređenje sa v1 je fer.
- Dodatna sekvenca je STIR, PSIR ili MP2RAGE (zavisno od pacijenta). Kad postoji samo T2, koristi se v1 → tako sistem radi i kada sekvenca nedostaje.


## Ćelija 0 — priprema + metrike

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q nnunetv2 nibabel SimpleITK

import os, glob, json, shutil, subprocess, csv
import numpy as np, nibabel as nib, SimpleITK as sitk
from scipy.ndimage import label
import torch

MASTER = '/content/drive/MyDrive/Master/kicma'
os.environ['nnUNet_raw']          = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{MASTER}/modeli/nnUNet_results'
for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]: os.makedirs(p, exist_ok=True)
REZ_DIR = f'{MASTER}/rezultati'; os.makedirs(REZ_DIR, exist_ok=True)
print('GPU:', torch.cuda.is_available())

STRUKT = np.ones((3,3,3), int)
def ucitaj(p): return np.asarray(nib.load(p).dataobj)

def dice(pred, gt):
    p = pred > 0; t = gt > 0
    if p.sum() + t.sum() == 0: return np.nan
    return float(2*(p & t).sum() / (p.sum() + t.sum()))

def lezijske(pred, gt):
    gl, ng = label(gt > 0, STRUKT); pl, npr = label(pred > 0, STRUKT)
    pm = pred > 0; gm = gt > 0
    tp = sum(1 for i in range(1, ng+1) if pm[gl == i].any())
    fp = sum(1 for j in range(1, npr+1) if not gm[pl == j].any())
    fn = ng - tp
    return dict(n_gt=ng, n_pred=npr, FP=fp,
                LTPR=tp/ng if ng else np.nan, LFPR=fp/npr if npr else 0.0,
                F1=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan)

def red(ime, pred, gt, **extra):
    L = lezijske(pred, gt)
    return dict(pacijent=ime, Dice=dice(pred, gt), vox_pred=int((pred > 0).sum()), **L, **extra)

def ci95(vals, B=2000, seed=0):
    v = np.array([x for x in vals if x == x])
    if len(v) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = [rng.choice(v, len(v), replace=True).mean() for _ in range(B)]
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def protokol(rows, naziv, ref=None, stampaj=True):
    """ MSSEG-2: Dice samo za pacijente SA lezijama;
       za pacijente BEZ lezija broje se lazne detekcije."""
    les = [r for r in rows if r['n_gt'] > 0]; bez = [r for r in rows if r['n_gt'] == 0]
    out = dict(skup=naziv, n_sa=len(les), n_bez=len(bez))
    if les:
        d = [r['Dice'] for r in les]; lo, hi = ci95(d)
        out.update(Dice=float(np.nanmean(d)), CI_lo=lo, CI_hi=hi,
                   LTPR=float(np.nanmean([r['LTPR'] for r in les])),
                   LFPR=float(np.nanmean([r['LFPR'] for r in les])),
                   F1=float(np.nanmean([r['F1'] for r in les])))
    if bez:
        fp = [r['n_pred'] for r in bez]
        out.update(FP_bez=float(np.mean(fp)), bez_cisto=int(sum(1 for x in fp if x == 0)))
    if stampaj:
        print(f"\n=== {naziv} ===")
        if les:
            print(f"  SA lezijama (n={len(les)}): Dice={out['Dice']:.3f} [95% CI {out['CI_lo']:.3f}-{out['CI_hi']:.3f}]  "
                  f"LTPR={out['LTPR']:.3f}  LFPR={out['LFPR']:.3f}  F1={out['F1']:.3f}"
                  + (f"   | baseline: {ref}" if ref is not None else ""))
        if bez:
            print(f"  BEZ lezija (n={len(bez)}): prosecno laznih lezija={out['FP_bez']:.2f}; "
                  f"bez ijedne lazne: {out['bez_cisto']}/{len(bez)}")
    return out

def u_csv(rows, path):
    if not rows: return
    keys = list(rows[0].keys())
    for r in rows:
        for k in r:
            if k not in keys: keys.append(k)
    with open(path, 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader()
        for r in rows: w.writerow(r)
    print('sacuvano:', path)

def svi_nii(folder):
    return sorted(glob.glob(f'{folder}/**/*.nii*', recursive=True))
def nadji(folder, *k, ne=()):
    fs = svi_nii(folder)
    for x in k:
        for f in fs:
            n = os.path.basename(f).lower()
            if x in n and not any(y in n for y in ne):
                return f
    return None
def fajlovi(sub_dir):
    """T2, maska i dodatna sekvenca (STIR/PSIR/MP2RAGE) jednog pacijenta."""
    if not os.path.isdir(sub_dir): return dict(t2=None, mask=None, extra=None, seq=None)
    maske = ('lesion', 'mask', 'seg', 'label')
    t2 = nadji(sub_dir, 't2', ne=maske)
    mask = nadji(sub_dir, 'lesionmask', 'lesion', 'mask')
    extra, seq = None, None
    for s in ['stir', 'psir', 'mp2rage', 'unit1']:
        f = nadji(sub_dir, s, ne=maske)
        if f: extra, seq = f, ('MP2RAGE' if s == 'unit1' else s.upper()); break
    return dict(t2=t2, mask=mask, extra=extra, seq=seq)

print('spremno')

Mounted at /content/drive
GPU: True
spremno


## Ćelija 1 — trener

In [ ]:
import nnunetv2
TR_NAME = 'nnUNetTrainer_100epochs_save10def'
vdir = os.path.join(os.path.dirname(nnunetv2.__file__), 'training', 'nnUNetTrainer', 'variants', 'training_length')
with open(os.path.join(vdir, f'{TR_NAME}.py'), 'w') as f:
    f.write("""import torch
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_100epochs
class nnUNetTrainer_100epochs_save10def(nnUNetTrainer_100epochs):
    # isto kao v1 (podrazumevani LR i oversampling); samo cuva checkpoint na 10 epoha
    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.save_every = 10
""")
print('trener:', TR_NAME)

trener: nnUNetTrainer_100epochs_save10def


## Ćelija 2 — raspakivanje + PROVERA strukture

In [ ]:
import tarfile
tar_path = f'{MASTER}/podaci_sirovi/MS_MultiSpine_dataset.tar.gz'
if not os.path.exists('/content/multispine'):
    print('raspakivam (nekoliko min)...')
    with tarfile.open(tar_path) as t: t.extractall('/content/multispine')
der = glob.glob('/content/multispine/**/derivatives', recursive=True)
assert der, 'Nema derivatives foldera u tar-u'
PAR = f'{der[0]}/preprocessedAndRegistered'
PRE = f'{der[0]}/preprocessed'
print('preprocessedAndRegistered:', os.path.isdir(PAR), '| preprocessed:', os.path.isdir(PRE))
SUBS = sorted(s for s in os.listdir(PAR) if s.startswith('sub-'))
print('pacijenata:', len(SUBS))

from collections import Counter
st = Counter()
for sub in SUBS:
    a = fajlovi(f'{PAR}/{sub}'); b = fajlovi(f'{PRE}/{sub}')
    st['PAR T2'] += bool(a['t2']); st['PAR maska'] += bool(a['mask']); st['PAR dodatna'] += bool(a['extra'])
    st['PRE T2'] += bool(b['t2']); st['PRE dodatna'] += bool(b['extra'])
print(dict(st))
print('sekvence (PAR):', Counter(fajlovi(f'{PAR}/{s}')['seq'] for s in SUBS))

def geo(p):
    if not p: return '-'
    im = sitk.ReadImage(p)
    return f"{os.path.basename(p)}  size={im.GetSize()} sp={tuple(round(x,2) for x in im.GetSpacing())}"
for sub in SUBS[:2]:
    a = fajlovi(f'{PAR}/{sub}'); b = fajlovi(f'{PRE}/{sub}')
    print(f'\n{sub}\n  PAR T2   : {geo(a["t2"])}\n  PAR maska: {geo(a["mask"])}\n  PAR dod. : {geo(a["extra"])}\n  PRE T2   : {geo(b["t2"])}\n  PRE dod. : {geo(b["extra"])}')

isti = 0; prov = 0
for sub in SUBS[:10]:
    a = fajlovi(f'{PAR}/{sub}')['t2']; b = fajlovi(f'{PRE}/{sub}')['t2']
    if not (a and b): continue
    A = sitk.ReadImage(a); B = sitk.ReadImage(b); prov += 1
    ok = (A.GetSize() == B.GetSize() and np.allclose(A.GetOrigin(), B.GetOrigin(), atol=1e-3)
          and np.allclose(A.GetSpacing(), B.GetSpacing(), atol=1e-4) and np.allclose(A.GetDirection(), B.GetDirection(), atol=1e-4))
    isti += ok
print(f'\nT2 iste geometrije u PAR i PRE: {isti}/{prov}  (ocekujemo sve)')

raspakivam (nekoliko min)...


/tmp/ipykernel_675/2856007610.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tar_path) as t: t.extractall('/content/multispine')


preprocessedAndRegistered: True | preprocessed: True
pacijenata: 100
{'PAR T2': 100, 'PAR maska': 100, 'PAR dodatna': 100, 'PRE T2': 100, 'PRE dodatna': 100}
sekvence (PAR): Counter({'STIR': 50, 'PSIR': 25, 'MP2RAGE': 25})

sub-001
  PAR T2   : 11-001_T2.nii.gz  size=(600, 600, 90) sp=(0.5, 0.5, 0.5)
  PAR maska: 11-001_LESIONMASK.nii.gz  size=(600, 600, 90) sp=(0.5, 0.5, 0.5)
  PAR dod. : 11-001_STIR.nii.gz  size=(600, 600, 90) sp=(0.5, 0.5, 0.5)
  PRE T2   : 11-001_T2.nii.gz  size=(600, 600, 90) sp=(0.5, 0.5, 0.5)
  PRE dod. : 11-001_STIR.nii.gz  size=(600, 600, 90) sp=(0.5, 0.5, 0.5)

sub-002
  PAR T2   : 11-002_T2.nii.gz  size=(760, 760, 96) sp=(0.5, 0.5, 0.5)
  PAR maska: 11-002_LESIONMASK.nii.gz  size=(760, 760, 96) sp=(0.5, 0.5, 0.5)
  PAR dod. : 11-002_STIR.nii.gz  size=(760, 760, 96) sp=(0.5, 0.5, 0.5)
  PRE T2   : 11-002_T2.nii.gz  size=(760, 760, 96) sp=(0.5, 0.5, 0.5)
  PRE dod. : 11-002_STIR.nii.gz  size=(760, 760, 96) sp=(0.5, 0.5, 0.5)

T2 iste geometrije u PAR i PRE: 10

## Ćelija 3 — nnU-Net format, 2 kanala — podesiv `VARIANT`
T2 i maska su **iste za oba kraka** (iz `preprocessedAndRegistered`, kao v1). Kanal 1 se uvek prebacuje na mrežu T2 snimka (nnU-Net traži istu geometriju kanala):
- `REG`: autorski registrovana dodatna sekvenca;
- `NOREG`: dodatna sekvenca iz `preprocessed` (neregistrovana), samo preuzorkovana na T2 mrežu (bez poravnavanja).

In [ ]:
VARIANT = 'NOREG'      #  'REG' ili 'NOREG'

DID, DNAME = (12, 'Dataset012_SpineREG') if VARIANT == 'REG' else (13, 'Dataset013_SpineNOREG')
DS = os.environ['nnUNet_raw'] + '/' + DNAME
shutil.rmtree(DS, ignore_errors=True)
for s in ['imagesTr', 'labelsTr']: os.makedirs(f'{DS}/{s}')

def na_t2_mrezu(src, t2_path, dst):
    fixed = sitk.ReadImage(t2_path); mov = sitk.ReadImage(src, sitk.sitkFloat32)
    out = sitk.Resample(mov, fixed, sitk.Transform(3, sitk.sitkIdentity), sitk.sitkLinear, 0.0, sitk.sitkFloat32)
    sitk.WriteImage(out, dst)

def upisi_masku(src, t2_path, dst):
    ref = sitk.ReadImage(t2_path); m = sitk.ReadImage(src)
    if m.GetSize() != ref.GetSize():
        raise ValueError('maska i T2 nisu iste velicine')
    arr = (sitk.GetArrayFromImage(m) > 0).astype(np.uint8)
    out = sitk.GetImageFromArray(arr); out.CopyInformation(ref)
    sitk.WriteImage(out, dst)

n = 0; preskoceno = []; seq_map = {}
for sub in SUBS:
    a = fajlovi(f'{PAR}/{sub}')
    extra = a['extra'] if VARIANT == 'REG' else fajlovi(f'{PRE}/{sub}')['extra']
    seq = a['seq'] or fajlovi(f'{PRE}/{sub}')['seq']
    if not (a['t2'] and a['mask'] and extra):
        preskoceno.append(sub); continue
    cid = sub.replace('-', '_')
    try:
        shutil.copy(a['t2'], f'{DS}/imagesTr/{cid}_0000.nii.gz')
        na_t2_mrezu(extra, a['t2'], f'{DS}/imagesTr/{cid}_0001.nii.gz')
        upisi_masku(a['mask'], a['t2'], f'{DS}/labelsTr/{cid}.nii.gz')
        seq_map[cid] = seq; n += 1
    except Exception as e:
        preskoceno.append(f'{sub} ({e})')

with open(f'{DS}/dataset.json', 'w') as f:
    json.dump({"channel_names": {"0": "T2", "1": "DODATNA"}, "labels": {"background": 0, "lesion": 1},
               "numTraining": n, "file_ending": ".nii.gz", "overwrite_image_reader_writer": "SimpleITKIO"}, f, indent=2)
from collections import Counter
print(f'{VARIANT}: upisano {n} pacijenata | po sekvenci: {dict(Counter(seq_map.values()))}')
print('preskoceno:', preskoceno if preskoceno else 'nijedan')
assert n > 0

NOREG: upisano 100 pacijenata | po sekvenci: {'STIR': 50, 'PSIR': 25, 'MP2RAGE': 25}
preskoceno: nijedan


## Ćelija 4

In [ ]:
!nnUNetv2_plan_and_preprocess -d {DID} --verify_dataset_integrity -npfp 2 -np 2

Fingerprint extraction...
Dataset013_SpineNOREG
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Extracting dataset fingerprint: 100% 100/100 [06:02<00:00,  3.62s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.515      0.51499997 0.515     ]. 
Current patch size: (np.int64(256), np.int64(64), np.int64(128)). 
Current median shape: [582.52427184  81.06796117 186.89320388]
Attempting to find 3d_lowres config. 
C

## Ćelija 5 — TRENING

In [ ]:
FOLD = 0
rez = f"{os.environ['nnUNet_results']}/{DNAME}/{TR_NAME}__nnUNetPlans__3d_fullres/fold_{FOLD}"
if os.path.exists(f"{rez}/checkpoint_final.pth"):
    print(f'{VARIANT} fold {FOLD} je VEC ZAVRSEN.')
else:
    nastavi = "--c" if os.path.exists(f"{rez}/checkpoint_latest.pth") else ""
    print(f'{VARIANT} fold {FOLD}:', "Nastavljam od checkpointa." if nastavi else "Krecem od nule.")
    !nnUNetv2_train {DID} 3d_fullres {FOLD} -tr {TR_NAME} {nastavi}

NOREG fold 0: Nastavljam od checkpointa.

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-11-13 10:30:52.939433: Using torch.compile...
2026-11-13 10:31:13.546063: do_d

## Ćelija 6 — POREĐENJE v1 / NOREG / REG

In [ ]:
FOLD_EVAL = 0
gt = {}; seq_of = {}
for sub in SUBS:
    a = fajlovi(f'{PAR}/{sub}')
    if a['mask']:
        cid = sub.replace('-', '_'); gt[cid] = a['mask']; seq_of[cid] = a['seq']

R = os.environ['nnUNet_results']
KRAKOVI = {
    'v1 (samo T2)':      f'{R}/Dataset011_Spine/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_{FOLD_EVAL}/validation',
    'NOREG (T2+dodatna)':f'{R}/Dataset013_SpineNOREG/{TR_NAME}__nnUNetPlans__3d_fullres/fold_{FOLD_EVAL}/validation',
    'REG (T2+dodatna)':  f'{R}/Dataset012_SpineREG/{TR_NAME}__nnUNetPlans__3d_fullres/fold_{FOLD_EVAL}/validation',
}
SVI = []; SAZ = []
for ime, vdir in KRAKOVI.items():
    if not os.path.isdir(vdir):
        print(f'\n{ime}: nema validation/ (trening nije zavrsen) -> preskacem'); continue
    rows = []
    for f in sorted(os.listdir(vdir)):
        if not f.endswith('.nii.gz'): continue
        cid = f.replace('.nii.gz', '')
        if cid not in gt: continue
        rows.append(red(cid, ucitaj(f'{vdir}/{f}'), ucitaj(gt[cid]) > 0, krak=ime, sekvenca=seq_of[cid]))
    SVI += rows
    SAZ.append(dict(krak=ime, sekvenca='SVE', **protokol(rows, f'{ime} — svi ({len(rows)})')))
    for s in ['STIR', 'PSIR', 'MP2RAGE']:
        sub_rows = [r for r in rows if r['sekvenca'] == s]
        if sub_rows: SAZ.append(dict(krak=ime, sekvenca=s, **protokol(sub_rows, f'{ime} — {s}')))

print('\n\nTABELA (pacijenti SA lezijama; Dice [95% CI]; lazne detekcije kod pacijenata BEZ lezija)')
print(f"{'krak':<22}{'sekv.':<9}{'n':>3}{'Dice':>7}{'95% CI':>15}{'LTPR':>7}{'LFPR':>7}{'FP bez':>8}")
for r in SAZ:
    if 'Dice' not in r: continue
    print(f"{r['krak']:<22}{r['sekvenca']:<9}{r['n_sa']:>3}{r['Dice']:>7.3f}{r['CI_lo']:>8.3f}-{r['CI_hi']:<6.3f}"
          f"{r['LTPR']:>7.3f}{r['LFPR']:>7.3f}{r.get('FP_bez', float('nan')):>8.2f}")
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
u_csv(SVI, f'{REZ_DIR}/kicma_v1_noreg_reg_po_pacijentu_{ts}.csv')
u_csv(SAZ, f'{REZ_DIR}/kicma_v1_noreg_reg_SAZETAK_{ts}.csv')


v1 (samo T2): nema validation/ (trening nije zavrsen) -> preskacem

=== NOREG (T2+dodatna) — svi (20) ===
  SA lezijama (n=15): Dice=0.493 [95% CI 0.398-0.581]  LTPR=0.881  LFPR=0.393  F1=0.667
  BEZ lezija (n=5): prosecno laznih lezija=3.20; bez ijedne lazne: 0/5

=== NOREG (T2+dodatna) — STIR ===
  SA lezijama (n=7): Dice=0.418 [95% CI 0.270-0.565]  LTPR=0.845  LFPR=0.300  F1=0.710
  BEZ lezija (n=4): prosecno laznih lezija=2.75; bez ijedne lazne: 0/4

=== NOREG (T2+dodatna) — PSIR ===
  SA lezijama (n=5): Dice=0.514 [95% CI 0.372-0.638]  LTPR=0.900  LFPR=0.612  F1=0.502

=== NOREG (T2+dodatna) — MP2RAGE ===
  SA lezijama (n=3): Dice=0.635 [95% CI 0.584-0.709]  LTPR=0.933  LFPR=0.244  F1=0.842
  BEZ lezija (n=1): prosecno laznih lezija=5.00; bez ijedne lazne: 0/1

=== REG (T2+dodatna) — svi (20) ===
  SA lezijama (n=15): Dice=0.465 [95% CI 0.362-0.570]  LTPR=0.803  LFPR=0.448  F1=0.573
  BEZ lezija (n=5): prosecno laznih lezija=2.40; bez ijedne lazne: 2/5

=== REG (T2+dodatna) — STI